# Section 06 lab — Execution and practical optimization

Each plan can be inspected without relying on an earlier exercise.


## Exercise 1 — Locate a shuffle in an aggregation plan


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F

events = spark.range(0, 100).withColumn('category', F.concat(F.lit('C'), (F.col('id') % 4)))
summary = events.groupBy('category').count()
summary.explain('formatted')
summary.orderBy("category").show()
print('Look for Exchange before the grouped result.')


## Exercise 2 — Inspect a broadcast join


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F

facts = spark.createDataFrame([(1, 'P1'), (2, 'P2'), (3, 'P1')], ['event_id', 'product_id'])
products = spark.createDataFrame([('P1', 'Accessories'), ('P2', 'Audio')], ['product_id', 'category'])
joined = facts.join(F.broadcast(products), 'product_id', 'left')
joined.explain('formatted')
joined.orderBy("event_id").show()
print('Look for BroadcastHashJoin in the plan.')


## Exercise 3 — Materialize and release a cache


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F

reused = spark.range(0, 100000).withColumn('bucket', F.col('id') % 10).cache()
print('Rows materialized:', reused.count())
reused.groupBy('bucket').count().show()
reused.agg(F.sum('id').alias('id_sum')).show()
reused.unpersist()
print('Cached after unpersist:', reused.is_cached)
assert reused.is_cached is False


## Exercise 4 — Repartition by a business key


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F

rows = [('O1', 'Accessories'), ('O2', 'Audio'), ('O3', 'Accessories'), ('O4', 'Electronics')]
orders = spark.createDataFrame(rows, ['order_id', 'category']).repartition(4, 'category')
distribution = (orders.withColumn('partition_id', F.spark_partition_id())
                .groupBy('partition_id', 'category').count()
                .orderBy('partition_id', 'category'))
distribution.show()
print('Partitions:', orders.rdd.getNumPartitions())
